In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from datasets import load_dataset, load_from_disk
from torch.utils.data import DataLoader

from model import NNUE
from dataset import transform_row, transform_batch, nnue_collate_fn

/home/josephyue/csci567/chess-engine/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(device)

cuda


In [3]:
ds = load_dataset(
    "mateuszgrzyb/lichess-stockfish-normalized",
    split="train", 
    streaming=True)

original_columns = ds.column_names
transformed_ds = ds.map(
    transform_row,
    remove_columns=original_columns
).filter(lambda x: x is not None)

In [4]:
# Take a small sample and manually iterate
sample = transformed_ds.take(100)
for i, item in enumerate(sample):
    if item is None:
        print(f"Row {i} is None!")
    # If item is a dict, check the values
    elif isinstance(item, dict):
        for k, v in item.items():
            if v is None:
                print(f"Row {i} has None in key: {k}")

In [ ]:
# Initialize Model, Loss, and Optimizer
model = NNUE().to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
# optimizer = optim.Adam(model.parameters(), lr=5e-4)

# SAVE_POINT = 50000

checkpoint = torch.load(f"nnue_checkpoints/chess_model_final.pt")
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [5]:
# Setup Splitting
shuffled_ds = transformed_ds.shuffle(seed=42, buffer_size=10000)

val_ds = shuffled_ds.take(2048) 
train_ds = shuffled_ds.skip(2048 + 1024 * 0)

val_loader = DataLoader(val_ds, batch_size=1024, num_workers=0, collate_fn=nnue_collate_fn)
train_loader = DataLoader(train_ds, batch_size=4096, num_workers=2, collate_fn=nnue_collate_fn, pin_memory=True)

In [6]:
# Training Hyperparameters
VAL_INTERVAL = 2000  # Validate every 2000 batches
SAVE_INTERVAL = 10000 # Save checkpoint

In [7]:
model.train()
pbar = tqdm(enumerate(train_loader), desc="Training")

running_train_loss = 0.0
train_samples_since_val = 0

for step, batch in pbar:
    # Move data to GPU
    stm_idx = batch['stm_idx'].to(device)
    stm_off = batch['stm_off'].to(device)
    nstm_idx = batch['nstm_idx'].to(device)
    nstm_off = batch['nstm_off'].to(device)
    targets = batch['target'].to(device)
    batch_size = targets.size(0)

    # Forward pass
    outputs = model(stm_idx, stm_off, nstm_idx, nstm_off)
    loss = criterion(outputs, targets)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_train_loss += loss.item() * batch_size
    train_samples_since_val += batch_size

    # --- Validation Loop ---
    if step % VAL_INTERVAL == 0 and step > 0:
        avg_train_loss = running_train_loss / train_samples_since_val

        model.eval()
        val_loss = 0.0
        total_val_samples = 0

        with torch.no_grad():
            for v_batch in val_loader:
                v_stm_idx = v_batch['stm_idx'].to(device)
                v_stm_off = v_batch['stm_off'].to(device)
                v_nstm_idx = v_batch['nstm_idx'].to(device)
                v_nstm_off = v_batch['nstm_off'].to(device)
                vt = v_batch['target'].to(device)

                current_batch_size = vt.size(0)

                v_out = model(v_stm_idx, v_stm_off, v_nstm_idx, v_nstm_off)
                loss = criterion(v_out, vt)
                val_loss += loss.item() * current_batch_size
                total_val_samples += current_batch_size
        
        if total_val_samples > 0:
            avg_val_loss = val_loss / total_val_samples
            # --- PRINT BOTH SIDE BY SIDE ---
            print(f"\n[Step {step}] Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        else:
            print(f"\nStep {step} | Validation produced no samples.")
        running_train_loss = 0.0
        train_samples_since_val = 0
        model.train()

    # --- Checkpointing ---
    if step % SAVE_INTERVAL == 0 and step > 0:
        save_loc = f"nnue_checkpoints/chess_model_step_{step}.pt"
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
        }, save_loc)
        print(f"Saved model to {save_loc}")

Training: 2001it [11:44,  1.16it/s]


[Step 2000] Train Loss: 0.11310 | Val Loss: 0.18507


Training: 4001it [23:04,  1.05s/it]


[Step 4000] Train Loss: 0.10819 | Val Loss: 0.19565


Training: 6001it [34:26,  1.08it/s]

...

Training: 74001it [7:00:25,  1.21it/s]


[Step 74000] Train Loss: 0.07434 | Val Loss: 0.20503


Training: 76001it [7:10:03,  1.06it/s]


[Step 76000] Train Loss: 0.13518 | Val Loss: 0.20316


Training: 77167it [7:20:40,  2.92it/s]


In [8]:
torch.save({
    'step': step,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item(),
}, f"nnue_checkpoints/chess_model_final.pt")